# CNN Image Classification Project - Final Colab Version

Clean final notebook for Google Colab using:

`/content/drive/MyDrive/AI 6th Sem/Course Work/archive.zip`

Built to avoid earlier problems: slow epochs, overfitting, rising validation loss, and slow transfer learning.

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Imports and Configuration

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

SEED = 42
tf.keras.utils.set_random_seed(SEED)

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.20

EPOCHS_BASELINE = 30
EPOCHS_DEEP = 30
EPOCHS_OPTIMIZER = 12
EPOCHS_TRANSFER_FEATURE_EXTRACTION = 4
EPOCHS_TRANSFER_FINE_TUNING = 3

AUTOTUNE = tf.data.AUTOTUNE

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 3: Extract Dataset

In [ ]:
zip_path = Path('/content/drive/MyDrive/AI 6th Sem/Course Work/archive.zip')
extract_dir = Path('/content/rice_leaf_dataset')
extract_dir.mkdir(parents=True, exist_ok=True)

if not zip_path.exists():
    raise FileNotFoundError(f'Zip file not found: {zip_path}')

if not any(extract_dir.iterdir()):
    print('Extracting dataset...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
else:
    print('Dataset already extracted.')

dataset_dir = extract_dir / 'Rice_Leaf_AUG'

if not dataset_dir.exists():
    possible_dirs = []
    for folder in extract_dir.rglob('*'):
        if folder.is_dir():
            child_dirs = [child for child in folder.iterdir() if child.is_dir()]
            if len(child_dirs) >= 2:
                possible_dirs.append(folder)
    if not possible_dirs:
        raise RuntimeError('Could not find class folders after extraction.')
    dataset_dir = possible_dirs[0]

print('Dataset directory:', dataset_dir)
print('Classes:')
for folder in sorted([p for p in dataset_dir.iterdir() if p.is_dir()]):
    print('-', folder.name)

## Cell 4: Load and Preprocess Data

In [ ]:
raw_train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir, validation_split=VALIDATION_SPLIT, subset='training',
    seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, label_mode='int'
)

raw_validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir, validation_split=VALIDATION_SPLIT, subset='validation',
    seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, label_mode='int'
)

class_names = raw_train_dataset.class_names
num_classes = len(class_names)
print('Class names:', class_names)
print('Number of classes:', num_classes)

normalization_layer = layers.Rescaling(1.0 / 255)

def normalize_images(images, labels):
    return normalization_layer(images), labels

train_dataset = (
    raw_train_dataset
    .map(normalize_images, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000, seed=SEED)
    .prefetch(AUTOTUNE)
)

validation_dataset = (
    raw_validation_dataset
    .map(normalize_images, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)

## Cell 5: Visualize Sample Images

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    for i in range(min(9, images.shape[0])):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 6: Class Distribution

In [ ]:
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
class_counts = {}
for class_name in class_names:
    class_folder = dataset_dir / class_name
    class_counts[class_name] = sum(
        1 for file_path in class_folder.rglob('*')
        if file_path.suffix.lower() in image_extensions
    )

plt.figure(figsize=(10, 5))
plt.bar(class_counts.keys(), class_counts.values(), color='steelblue')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(class_counts)

## Cell 7: Helper Functions

In [ ]:
def make_callbacks(patience=4):
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=patience, restore_best_weights=True
    )
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.2, patience=max(1, patience // 2),
        min_lr=1e-6, verbose=1
    )
    return [early_stopping, reduce_lr]

def plot_training_history(history, title):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, label='Training Accuracy')
    plt.plot(epochs, val_acc, label='Validation Accuracy')
    plt.title(f'{title}: Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, label='Training Loss')
    plt.plot(epochs, val_loss, label='Validation Loss')
    plt.title(f'{title}: Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def collect_predictions(model, dataset):
    y_true_batches, y_pred_batches = [], []
    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        predictions = np.argmax(probabilities, axis=1)
        y_true_batches.append(labels.numpy())
        y_pred_batches.append(predictions)
    return np.concatenate(y_true_batches), np.concatenate(y_pred_batches)

def evaluate_model(model, dataset, class_names, title):
    loss, accuracy = model.evaluate(dataset, verbose=0)
    y_true, y_pred = collect_predictions(model, dataset)

    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f'{title} Results')
    print('-' * 40)
    print(f'Validation Loss:      {loss:.4f}')
    print(f'Validation Accuracy:  {accuracy:.4f}')
    print(f'Weighted Precision:   {precision:.4f}')
    print(f'Weighted Recall:      {recall:.4f}')
    print(f'Weighted F1-score:    {f1:.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
    plt.title(f'{title}: Confusion Matrix')
    plt.tight_layout()
    plt.show()

    return {'loss': loss, 'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

def print_results_table(results):
    print(f"{'Model':<35} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-score':>10}")
    print('-' * 80)
    for name, m in results.items():
        print(f"{name:<35} {m['accuracy']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}")

## Cell 8: Baseline CNN From Scratch

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name='data_augmentation')

def build_baseline_cnn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.50),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.30),
        layers.Dense(32, activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ], name='improved_baseline_cnn')
    return model

baseline_model = build_baseline_cnn(IMAGE_SIZE + (3,), num_classes)
baseline_model.summary()

## Cell 9: Train and Evaluate Baseline CNN

In [ ]:
baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

baseline_history = baseline_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_BASELINE,
    callbacks=make_callbacks(patience=4)
)

plot_training_history(baseline_history, 'Improved Baseline CNN')
baseline_results = evaluate_model(baseline_model, validation_dataset, class_names, 'Improved Baseline CNN')

## Cell 10: Deeper CNN

In [ ]:
def build_deeper_cnn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,

        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.30),

        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.35),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.50),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.30),
        layers.Dense(num_classes, activation='softmax'),
    ], name='deeper_cnn')
    return model

deep_model = build_deeper_cnn(IMAGE_SIZE + (3,), num_classes)
deep_model.summary()

## Cell 11: Train and Evaluate Deeper CNN

In [ ]:
deep_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

deep_history = deep_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_DEEP,
    callbacks=make_callbacks(patience=4)
)

plot_training_history(deep_history, 'Deeper CNN')
deep_results = evaluate_model(deep_model, validation_dataset, class_names, 'Deeper CNN')

model_results = {
    'Improved Baseline CNN': baseline_results,
    'Deeper CNN': deep_results
}
print_results_table(model_results)

## Cell 12: Optimizer Comparison - SGD vs Adam

In [ ]:
def train_deeper_with_optimizer(optimizer, optimizer_name):
    model = build_deeper_cnn(IMAGE_SIZE + (3,), num_classes)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f'Training with {optimizer_name}...')
    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS_OPTIMIZER,
        callbacks=make_callbacks(patience=3),
        verbose=1
    )
    return model, history

sgd_model, sgd_history = train_deeper_with_optimizer(
    keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'SGD with momentum'
)

adam_model, adam_history = train_deeper_with_optimizer(
    keras.optimizers.Adam(learning_rate=0.001),
    'Adam'
)

## Cell 13: Plot Optimizer Comparison

In [ ]:
optimizer_histories = {'SGD': sgd_history, 'Adam': adam_history}

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
for name, history in optimizer_histories.items():
    plt.plot(history.history['val_accuracy'], label=f'{name} Validation Accuracy')
plt.title('Optimizer Comparison: Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for name, history in optimizer_histories.items():
    plt.plot(history.history['val_loss'], label=f'{name} Validation Loss')
plt.title('Optimizer Comparison: Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for name, history in optimizer_histories.items():
    best_acc = max(history.history['val_accuracy'])
    best_epoch = int(np.argmax(history.history['val_accuracy']) + 1)
    final_loss = history.history['val_loss'][-1]
    print(f'{name}: best val accuracy = {best_acc:.4f} at epoch {best_epoch}, final val loss = {final_loss:.4f}')

## Cell 14: Transfer Learning With Fast MobileNetV2

In [ ]:
def build_transfer_learning_model(input_shape, num_classes):
    base_model = keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet',
        alpha=0.35
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = layers.Lambda(
        lambda batch: keras.applications.mobilenet_v2.preprocess_input(batch * 255.0),
        name='mobilenetv2_preprocessing'
    )(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='fast_mobilenetv2_transfer')
    return model, base_model

transfer_model, transfer_base_model = build_transfer_learning_model(IMAGE_SIZE + (3,), num_classes)
transfer_model.summary()

## Cell 15: Feature Extraction

In [ ]:
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

transfer_feature_history = transfer_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_TRANSFER_FEATURE_EXTRACTION,
    callbacks=make_callbacks(patience=2)
)

plot_training_history(transfer_feature_history, 'Transfer Learning Feature Extraction')

## Cell 16: Fine-Tuning

In [ ]:
transfer_base_model.trainable = True
fine_tune_at = max(0, len(transfer_base_model.layers) - 10)

for layer in transfer_base_model.layers[:fine_tune_at]:
    layer.trainable = False

for layer in transfer_base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

transfer_fine_tune_history = transfer_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_TRANSFER_FINE_TUNING,
    callbacks=make_callbacks(patience=2)
)

plot_training_history(transfer_fine_tune_history, 'Transfer Learning Fine-Tuning')
transfer_results = evaluate_model(transfer_model, validation_dataset, class_names, 'MobileNetV2 Transfer Learning')

## Cell 17: Final Comparison

In [ ]:
model_results['MobileNetV2 Transfer Learning'] = transfer_results
print_results_table(model_results)

## Cell 18: Prediction / Inference

In [ ]:
best_model = transfer_model

plt.figure(figsize=(14, 14))
for images, labels in validation_dataset.take(1):
    probabilities = best_model.predict(images, verbose=0)
    predicted_labels = np.argmax(probabilities, axis=1)

    for i in range(min(9, images.shape[0])):
        actual_label = class_names[int(labels[i])]
        predicted_label = class_names[int(predicted_labels[i])]
        confidence = np.max(probabilities[i])

        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(f'Actual: {actual_label}\nPredicted: {predicted_label} ({confidence:.2f})', fontsize=10)
        plt.axis('off')

plt.suptitle('Sample Validation Predictions', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 19: Short Explanation

Baseline CNN uses three convolutional blocks, pooling, three Dense layers, Dropout, and Softmax. The model uses GlobalAveragePooling2D to reduce overfitting compared with a large Flatten layer.

The deeper CNN doubles the convolutional depth and adds Batch Normalization plus Dropout.

The optimizer comparison trains the same deeper architecture with SGD and Adam. Adam usually converges faster, while SGD may sometimes generalize well.

Transfer learning uses MobileNetV2 pretrained on ImageNet. The first stage freezes the base model, and the second stage fine-tunes only the last few layers with a low learning rate.